## 環境設置

In [1]:
try:
    import numpy as np  # noqa: F401
except ImportError:
    import sys
    !{sys.executable} -m pip install -q numpy

# Case-03.6c:應變相容法修正版強度檢核

依討論規劃,這裡用**應變相容法**(規範 P-M 交互作用圖背後的正規算法)
取代 Case-03.6 誤用的梁公式,**不修改 Case-03.6**——保留它當作「已知
限制」的歷史紀錄,問題與修正過程完整寫在 Case-06 跟 ROADMAP。

**這裡要解決兩件事**:
1. 提供一個真正適用於柱斷面(對稱配筋+軸力共同作用)的快速強度公式
2. 重新確認 Case-03.7、Case-04 的 Design Loop 收斂結果是否受影響

**受影響範圍先講清楚,不是每個 Case 都要補**:
- Case-03.7、Case-04:Design Loop 收斂到 20cm 這個結論**確實翻盤**
- Case-03.6b:用了問題公式,但示範用的 40cm 修正後仍 PASS,結論沒有翻盤
- Case-03.5、Case-04.5、Case-05:完全不涉及強度 PASS/FAIL,不受影響

## 第 1 課:通用版應變相容法函式

跟 Case-06 用的是同一套邏輯(平面保持平面、力平衡),但寫成可以吃
任意柱斷面尺寸的通用函式,不是只驗證一個 40cm 案例。

In [2]:
fc, fy, Es = 280.0, 4200.0, 2.0e6   # kgf/cm^2
beta1 = 0.85
eps_cu = 0.003
rho = 0.02   # 沿用一路假設的縱向鋼筋比


def Mn_strain_compat(h_col_m, Pu_kN):
    """應變相容法算標稱彎矩強度, h_col_m=柱邊長(m), Pu_kN=軸力(kN,壓力正值)"""
    h = h_col_m*100
    cover = 4.0
    half = h/2 - cover
    if half <= 0:
        return 0.0

    As_bar = (rho*h*h)/8
    bar_layers = {half: 3, 0.0: 2, -half: 3}   # 8根鋼筋: 壓力側3+中央2+拉力側3

    Pu = Pu_kN*1000/9.80665
    eps_y = fy/Es

    def section_force(c):
        a = min(beta1*c, h)
        Cc = 0.85*fc*h*a
        y_Cc = half - a/2
        N = Cc
        M = Cc*y_Cc
        for y, n_bars in bar_layers.items():
            As = n_bars*As_bar
            dist_from_top = half - y
            eps_s = eps_cu*(c-dist_from_top)/c if c > 0 else 0.0
            eps_s = max(min(eps_s, eps_y), -eps_y)
            fs = Es*eps_s
            fs_net = fs - 0.85*fc if dist_from_top <= a else fs
            Fs = As*fs_net
            N += Fs
            M += Fs*y
        return N, M

    c_lo, c_hi = 0.1, h*3
    for _ in range(100):
        c_mid = (c_lo+c_hi)/2
        N, M = section_force(c_mid)
        if N > Pu:
            c_hi = c_mid
        else:
            c_lo = c_mid
    c_final = (c_lo+c_hi)/2
    _, M_final = section_force(c_final)
    return M_final*9.80665e-5


Mn_40 = Mn_strain_compat(0.40, 147.60)
print(f"驗證40cm柱: Mn = {Mn_40:.2f} kN-m")

# 迴歸測試: 確認跟Case-06算出的結果一致
assert abs(Mn_40 - 220.83) < 0.1, "跟Case-06的應變相容法結果對不起來!"
print("[PASS] 與Case-06結果一致")

驗證40cm柱: Mn = 220.83 kN-m
[PASS] 與Case-06結果一致


## 第 2 課:逐尺寸比較——修正比例不是固定值

先看結果,再解釋為什麼重要。

In [3]:
print(f"{'柱尺寸':<8}{'應變相容法Mn':<16}{'原簡化公式Mn':<16}{'比例'}")
ratios = []
for h in [0.15,0.18,0.20,0.25,0.30,0.35,0.40]:
    Mn_sc = Mn_strain_compat(h, 147.60)
    b=d=h*100; Ag=b*d; As=rho*Ag; a=As*fy/(0.85*fc*b)
    Mn_orig = As*fy*(d-a/2)*9.80665e-5
    ratio = Mn_sc/Mn_orig
    ratios.append(ratio)
    print(f"{h*100:.0f}cm    {Mn_sc:<16.2f}{Mn_orig:<16.2f}{ratio:.3f}")

# 驗證: 比例不是常數, 越小尺寸誤差越嚴重
assert ratios[0] < ratios[-1], "小尺寸的高估比例應該比大尺寸更嚴重"
print(f"\n[確認] 修正比例從{ratios[0]:.3f}(15cm)到{ratios[-1]:.3f}(40cm), 不是固定值")
print("越小的柱子, 原公式高估越嚴重——因為軸力佔斷面容量的比例越大,")
print("越推向「全部鋼筋當拉力鋼筋」這個簡化假設最失真的區間")

柱尺寸     應變相容法Mn         原簡化公式Mn         比例
15cm    6.50            22.90           0.284
18cm    14.71           39.56           0.372
20cm    21.88           54.27           0.403
25cm    48.44           106.00          0.457
30cm    89.11           183.17          0.486
35cm    145.29          290.86          0.500
40cm    220.83          434.17          0.509

[確認] 修正比例從0.284(15cm)到0.509(40cm), 不是固定值
越小的柱子, 原公式高估越嚴重——因為軸力佔斷面容量的比例越大,
越推向「全部鋼筋當拉力鋼筋」這個簡化假設最失真的區間


## 第 3 課:重新跑 Design Loop——找出真正的 governing 尺寸

沿用 Case-03.7 的 `design_loop` 架構,只換掉強度檢核函式。位移角部分
完全不受這次發現影響,直接沿用 Case-03.6 算過的數字。

In [4]:
DRIFT_LIMIT = 0.005
phi_moment = 0.65
Mu = 22.608   # kN-m, demand端完全不變

# 位移角利用率沿用Case-03.6原始算出的數字(這部分邏輯沒有問題)
drift_util_table = {0.15:2.538, 0.18:1.224, 0.20:0.803, 0.25:0.329,
                     0.30:0.159, 0.35:0.086, 0.40:0.050}

print(f"{'柱尺寸':<8}{'drift結果':<10}{'修正M利用率':<14}{'強度結果':<10}{'總結'}")
governing_size = None
for h in [0.15,0.18,0.20,0.25,0.30,0.35,0.40]:
    Mn_sc = Mn_strain_compat(h, 147.60)
    phiMn = phi_moment*Mn_sc
    m_util = Mu/phiMn
    drift_ok = drift_util_table[h] < 1.0
    strength_ok = m_util < 1.0
    overall = drift_ok and strength_ok
    if overall and governing_size is None:
        governing_size = h
    print(f"{h*100:.0f}cm    {'PASS' if drift_ok else 'FAIL':<10}{m_util:<14.1%}"
          f"{'PASS' if strength_ok else 'FAIL':<10}{'PASS' if overall else 'FAIL'}")

print(f"\n真正的governing尺寸: {governing_size*100:.0f}cm")

assert governing_size == 0.25, f"預期25cm是真正governing尺寸, 實際{governing_size}"
print("[PASS] 確認25cm是修正後真正通過雙重檢核的最小尺寸")

print(f"\n=== 與Case-03.7/Case-04原始結論對照 ===")
print(f"原始Design Loop收斂: 20cm (基於高估的強度公式)")
print(f"修正後真正governing: 25cm")
print(f"20cm在修正後的強度利用率: 159.0% (FAIL)")

柱尺寸     drift結果   修正M利用率        強度結果      總結
15cm    FAIL      534.9%        FAIL      FAIL
18cm    FAIL      236.5%        FAIL      FAIL
20cm    PASS      159.0%        FAIL      FAIL
25cm    PASS      71.8%         PASS      PASS
30cm    PASS      39.0%         PASS      PASS
35cm    PASS      23.9%         PASS      PASS
40cm    PASS      15.8%         PASS      PASS

真正的governing尺寸: 25cm
[PASS] 確認25cm是修正後真正通過雙重檢核的最小尺寸

=== 與Case-03.7/Case-04原始結論對照 ===
原始Design Loop收斂: 20cm (基於高估的強度公式)
修正後真正governing: 25cm
20cm在修正後的強度利用率: 159.0% (FAIL)


## 總結表

In [5]:
print("="*55)
print("Case-03.6c 應變相容法修正版強度檢核總結")
print("="*55)
print(f"{'40cm柱驗證':<24}Mn=220.83kN-m(與Case-06一致)")
print(f"{'原公式高估比例範圍':<24}{ratios[0]:.1%}~{ratios[-1]:.1%}(隨尺寸變動)")
print(f"{'真正governing尺寸':<24}{governing_size*100:.0f}cm")
print(f"{'受影響: Case-03.7':<24}Design Loop結論20cm->25cm")
print(f"{'受影響: Case-04':<24}Design Loop結論20cm->25cm")
print(f"{'不受影響':<24}Case-03.5/03.6b(40cm結論不變)/04.5/05")
print()
print("Case-03.6c [PASS] -- 修正版強度檢核建立完成")

Case-03.6c 應變相容法修正版強度檢核總結
40cm柱驗證                 Mn=220.83kN-m(與Case-06一致)
原公式高估比例範圍               28.4%~50.9%(隨尺寸變動)
真正governing尺寸           25cm
受影響: Case-03.7          Design Loop結論20cm->25cm
受影響: Case-04            Design Loop結論20cm->25cm
不受影響                    Case-03.5/03.6b(40cm結論不變)/04.5/05

Case-03.6c [PASS] -- 修正版強度檢核建立完成
